In [35]:
import json

T = CartanType(['A', 3])
W = WeylGroup(T,prefix="s")
[s1, s2, s3] = W.simple_reflections()
KLP = json.load( open( "kl_polys/a3.json" ) )

WCR = WeylCharacterRing(T, style="coroots", base_ring=QQ)
R = WeightRing(WCR)
# EXPI MODIFIED, THIS CODE ONLY WORKS IN TYPE A
f = [w.coerce_to_sl() for w in WCR.fundamental_weights()]

n = len(W.simple_reflections())
e = s1^2
w0 = W.long_element()
S.<q> = LaurentPolynomialRing(QQ)
KL = KazhdanLusztigPolynomial(W,q)
q = 11

def Iw(w):
    a = []
    for i in range(1, n+1):
        if w.has_right_descent(i):
            a.append(i)
    return a

def Iwc(w):
    a = []
    for i in range(1, n+1):
        if not w.has_right_descent(i):
            #a.append(W.simple_reflections()[i])
            a.append(i)
    return a

def expI(l):
    a = R.one()
    for i in l:
        a = a * R(f[i-1])
    return a

def Cpsub_on_wr_element(w, v):
    interv = W.bruhat_interval(e, w)
    s = 0
    for y in interv:
        a = v.weyl_group_action(y)
        s += int(KLP[str((y, w))]) * a
    return s

def Cpsup_on_wr_element(w, v):
    interv = W.bruhat_interval(e, w0*w)
    s = 0
    for y in interv:
        a = v.weyl_group_action(w0*y)
        s += int(KLP[str((y, w0*w))]) * a
    return s

def fsub(w):
    return Cpsub_on_wr_element(w, expI(Iwc(w)))

def qfsub(w):
    return Cpsub_on_wr_element(w, expI(Iwc(w)).scale(q))

def fsup(w):
    return Cpsup_on_wr_element(w, expI(Iw(w)))

invols = []
for w in W:
    if w*w == e:
        invols.append(w)

def Iweight(w):
    wgt = 0
    for a in Iw(w):
        wgt += f[a-1]
    return wgt

rho = Iweight(w0)

def weightpairing(l):
    a = WCR.dot_reduce(l - rho)
    if a[0] == 0:
        return 0
    else:
        return a[0]*WCR(a[1])

def woke_pairing(a, b):
    #takes two elements of R and gives you an element of WCR
    d = dict(R(a*b))
    r = 0
    for l in d:
        r += d[l]*weightpairing(l)
    return r

invols = []
for w in W:
    if w*w == e:
        invols.append(w)

d_sub = {}
d_sup = {}

for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    d_sub[i] = fsub(W[i])
    d_sup[i] = fsup(W[i])

print("Setup done, f_w, f^w are defined.")

Setup done, f_w, f^w are defined.


In [30]:
#want to compare svk_dict to fsup. svk_dual_dict is not properly normalized for woke_pairing!
svk_dict = {s1*s2*s3*s1*s2*s1: a3(-1,-1,-1), s1*s2*s3*s1*s2: a3(0,-1,-1), s1*s2*s3*s2*s1: a3(-1,0,-1), s2*s3*s1*s2*s1: a3(-1,-1,0), s1*s2*s3*s2: a3(-1,0,-1) + a3(1,-1,-1), s1*s2*s3*s1: a3(0,-1,-1) + a3(-1,1,-2) + a3(-1,0,0), s2*s3*s2*s1: a3(-1,-1,0) + a3(-2,1,-1) + a3(0,0,-1), s2*s3*s1*s2: a3(0,-1,0), s3*s1*s2*s1: a3(-1,-1,1) + a3(-1,0,-1), s2*s3*s2: a3(-1,-1,0) + a3(-2,1,-1) + a3(1,-2,0) + 2*a3(0,0,-1) + a3(0,-1,1) + a3(2,-1,-1), s1*s2*s3: a3(0,0,-1), s2*s3*s1: a3(0,-1,0) + a3(-1,1,-1), s3*s1*s2: a3(0,0,0) + a3(-1,-1,1) + a3(-1,0,-1) + a3(1,-2,1) + a3(1,-1,-1), s1*s2*s1: a3(0,-1,-1) + a3(0,-2,1) + a3(-1,1,-2) + 2*a3(-1,0,0) + a3(-1,-1,2) + a3(1,-1,0), s3*s2*s1: a3(-1,0,0), s2*s3: a3(0,-1,0) + a3(-1,1,-1) + a3(1,0,-1), s3*s1: 3*a3(0,0,0) + a3(-2,1,0) + a3(-1,-1,1) + a3(-1,0,-1) + a3(1,-2,1) + a3(1,-1,-1) + a3(-1,2,-1) + a3(0,1,-2), s3*s2: a3(-1,0,0) + a3(1,-1,0), s1*s2: a3(0,0,-1) + a3(0,-1,1), s2*s1: a3(0,-1,0) + a3(-1,1,-1) + a3(-1,0,1), s3: a3(-1,0,0) + a3(1,-1,0) + a3(0,1,-1), s2: a3(0,-1,0) + a3(-1,1,-1) + a3(-1,0,1) + a3(1,0,-1) + a3(1,-1,1), s1: a3(0,0,-1) + a3(0,-1,1) + a3(-1,1,0), s1*s1: a3(0,0,0)}
svk_dual_dict = {s1*s2*s3*s1*s2*s1: a3(1,1,1), s1*s2*s3*s1*s2: -a3(1,1,0) - a3(1,0,2) - a3(0,2,1), s1*s2*s3*s2*s1: -a3(0,2,0) - a3(0,1,2) - a3(1,0,1) - a3(2,1,0) - a3(2,0,2), s2*s3*s1*s2*s1: -a3(0,1,1) - a3(2,0,1) - a3(1,2,0), s1*s2*s3*s2: a3(0,2,0) + a3(0,1,2) + a3(1,0,1), s1*s2*s3*s1: a3(1,1,0) + a3(1,0,2), s2*s3*s2*s1: a3(0,1,1) + a3(2,0,1), s2*s3*s1*s2: a3(0,1,0) + a3(0,0,2) + a3(-1,2,1) + a3(2,0,0) + a3(2,-1,2) + a3(1,2,-1) + 3*a3(1,1,1) + a3(0,3,0), s3*s1*s2*s1: a3(0,2,0) + a3(1,0,1) + a3(2,1,0), s2*s3*s2: -a3(0,1,1), s1*s2*s3: -a3(1,0,0) - a3(1,-1,2) - a3(0,2,-1) - 2*a3(0,1,1) - a3(0,0,3) - a3(2,0,1), s2*s3*s1: -a3(0,1,0) - a3(0,0,2) - a3(2,0,0) - a3(2,-1,2) - a3(1,1,1), s3*s1*s2: -a3(0,2,0) - a3(1,0,1), s1*s2*s1: -a3(1,1,0), s3*s2*s1: -a3(0,0,1) - a3(-1,2,0) - a3(2,-1,1) - 2*a3(1,1,0) - a3(1,0,2) - a3(3,0,0), s2*s3: a3(0,1,0) + a3(0,0,2), s3*s1: a3(1,0,1), s3*s2: a3(0,0,1) + a3(-1,2,0) + a3(1,1,0), s1*s2: a3(1,0,0) + a3(0,2,-1) + a3(0,1,1), s2*s1: a3(0,1,0) + a3(2,0,0), s3: -a3(0,0,1), s2: -a3(0,1,0), s1: -a3(1,0,0), s1*s1: a3(0,0,0)}

In [46]:
svk_dual = {}
for i in range(len(W)):
    svk_dual[i] = svk_dual_dict[W[i]]

print(svk_dual)

{0: a3(0,0,0), 1: a3(1,0,1), 2: a3(0,1,0) + a3(0,0,2) + a3(-1,2,1) + a3(2,0,0) + a3(2,-1,2) + a3(1,2,-1) + 3*a3(1,1,1) + a3(0,3,0), 3: a3(1,1,1), 4: a3(1,0,0) + a3(0,2,-1) + a3(0,1,1), 5: a3(1,1,0) + a3(1,0,2), 6: a3(0,0,1) + a3(-1,2,0) + a3(1,1,0), 7: a3(0,1,1) + a3(2,0,1), 8: a3(0,1,0) + a3(2,0,0), 9: a3(0,1,0) + a3(0,0,2), 10: a3(0,2,0) + a3(0,1,2) + a3(1,0,1), 11: a3(0,2,0) + a3(1,0,1) + a3(2,1,0), 12: -a3(1,0,0), 13: -a3(0,0,1), 14: -a3(1,1,0) - a3(1,0,2) - a3(0,2,1), 15: -a3(0,1,1) - a3(2,0,1) - a3(1,2,0), 16: -a3(0,1,0), 17: -a3(0,1,0) - a3(0,0,2) - a3(2,0,0) - a3(2,-1,2) - a3(1,1,1), 18: -a3(0,2,0) - a3(1,0,1), 19: -a3(0,2,0) - a3(0,1,2) - a3(1,0,1) - a3(2,1,0) - a3(2,0,2), 20: -a3(1,1,0), 21: -a3(1,0,0) - a3(1,-1,2) - a3(0,2,-1) - 2*a3(0,1,1) - a3(0,0,3) - a3(2,0,1), 22: -a3(0,1,1), 23: -a3(0,0,1) - a3(-1,2,0) - a3(2,-1,1) - 2*a3(1,1,0) - a3(1,0,2) - a3(3,0,0)}


In [50]:
def build_matrix():
    m = []
    for i in range(len(W)):
        print(i + 1, "out of", len(W), end='\r')
        r = []
        for j in range(len(W)):
            r.append(woke_pairing(d_sup[i], d_sub[j]))
        m.append(r)
    return m
bigmat = build_matrix()
#bigmat[i][j] = woke_pairing(d_sup[i], d_sub[j]) for all i, j
alls = list(range(len(W)))
pops = list(range(len(W)))
order = []
while len(pops) > 0:
    for a in pops:
        good = True
        for b in pops:
            if b != a and bigmat[b][a] != 0:
                good = False
        if good:
            order.append(a)
            pops.remove(a)

order.reverse()
print("Computed the matrix of pairings and the upper triangular order.")

Computed the matrix of pairings and the upper triangular order.


In [41]:
def lc(i):
    a = bigmat[i][i]
    d = dict(WCR(a))
    if len(d) != 1:
        print("LC ERROR")
    else:
        return list(d.values())[0]

counter = 0
d_dual = {}
for j in order:
    counter += 1
    print(counter, "out of", len(W), end='\r')
    d_dual[j] = d_sub[j]/lc(j)
    for i in order:
        if  i != j and woke_pairing(d_sup[i], d_dual[j]) != 0:
            d_dual[j] = d_dual[j] - woke_pairing(d_sup[i], d_dual[j])*d_sub[i]/lc(i)
print("Computed the dual basis as the dictionary d_dual.")

Computed the dual basis as the dictionary d_dual.


In [44]:
#OPTIONAL CELL: CHECKING THE DUAL BASIS IS VALID.
valid = True
for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    for j in range(len(W)):
        a = woke_pairing(d_sup[i], d_dual[j])
        if a != 0:
            if i != j or a != 1:
                print("ISSUE AT", i, j)
                valid = False
if valid:
    print("Checked dual basis, all is good.")
else:
    print("ISSUES! Dual basis is NOT correct.")

Checked dual basis, all is good.


In [43]:
print(d_dual)

{14: -1/2*a3(0,-1,1) - 1/2*a3(-1,1,0) - 1/2*a3(1,0,0), 21: -1/6*a3(-2,1,1) - 1/3*a3(0,0,1) - 1/6*a3(-1,2,0) - 1/6*a3(2,-1,1) - 1/6*a3(1,1,0), 6: 1/4*a3(1,1,-1) + 1/4*a3(1,0,1), 18: -1/4*a3(0,0,0) - 1/4*a3(1,1,-1) - 1/4*a3(1,0,1) - 1/4*a3(-1,2,-1) - 1/4*a3(-1,1,1), 10: 1/2*a3(-1,1,0) + 1/2*a3(1,0,0), 2: 3/4*a3(0,0,0) + 1/4*a3(2,-1,0) + 1/4*a3(1,1,-1) + 1/2*a3(1,0,1) + 1/4*a3(1,-2,1) + 1/4*a3(-1,2,-1) + 1/4*a3(-1,1,1) + 1/4*a3(0,-1,2), 23: -1/6*a3(1,1,-2) - 1/3*a3(1,0,0) - 1/6*a3(1,-1,2) - 1/6*a3(0,2,-1) - 1/6*a3(0,1,1), 20: -1/2*a3(0,0,1), 16: -1/4*a3(1,0,1), 12: -1/6*a3(0,1,1), 8: 1/6*a3(1,-1,2) + 1/6*a3(0,1,1), 4: 1/4*a3(1,0,1) + 1/4*a3(-1,1,1), 0: -1/24*a3(1,-1,1) + 1/12*a3(0,1,0) + 1/24*a3(1,1,1), 22: -1/2*a3(1,0,0), 19: -1/2*a3(-1,1,-1) - 1/2*a3(-1,0,1) - 1/2*a3(1,0,-1) - 1/2*a3(1,-1,1) - a3(0,1,0), 17: -1/2*a3(1,-1,1) - 1/2*a3(0,1,0), 15: -1/2*a3(1,-1,0) - 1/2*a3(0,1,-1) - 1/2*a3(0,0,1), 13: -1/6*a3(1,1,0), 11: 1/2*a3(0,1,-1) + 1/2*a3(0,0,1), 9: 1/6*a3(2,-1,1) + 1/6*a3(1,1,0), 7: 

In [52]:
def build_matrix():
    m = []
    for i in range(len(W)):
        print(i + 1, "out of", len(W), end='\r')
        r = []
        for j in range(len(W)):
            r.append(woke_pairing(d_sup[i], svk_dual[j]))
        m.append(r)
    return m
bigmat = build_matrix()
#bigmat[i][j] = woke_pairing(d_sup[i], d_sub[j]) for all i, j
print(matrix(bigmat))

[                        0                         0                         0              24*A3(0,0,0)                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0]
[                        0                         0               2*A3(0,2,0)              -6*A3(0,0,0)               2*A3(0,0,1)                         0               2*A3(1,0,0)                         0               2*A3(0,0,0)               2*A3(0,0,0)                         0                         0                         0                         0          

In [54]:
bigmat[18]

18

In [55]:
newmat = []
for i in range(len(W)):
    newrow = []
    for j in range(len(W)):
        newrow.append(bigmat[order[i]][order[j]])
    newmat.append(newrow)

print(matrix(newmat))

[                        0               2*A3(0,0,0)                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0]
[              6*A3(0,0,0)                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0                         0          